# GO Enrichment of Dysbiosis DE Genes, Split by Namespace (Recalled Genera)

Updated version of `Local_analysis/Transcriptome_vs_weather_and_microbiome/GO_term_enrichment/go_term_analysis_split_by_molecular_function_and_location.ipynb`.

It uses the significant genes (padj < 0.05) and background genes from `transcriptomic_impact_of_dysbiosis_recalled_genera.ipynb`. Run that notebook first. Enrichment is run separately for biological process, molecular function and cellular component, as in the original.

If the DE notebook finds no significant genes, the enrichment cells print a message and skip.

In [1]:
import pandas as pd
import json
import Go_tools
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
gobasic = json.load(open("/Users/michael/Data/Reference_data/go-basic.json", "r"))

In [3]:
go_term_matrix = pd.read_csv(
    "/Users/michael/Data/Reference_data/Arabdidopsis_go_matrix_from_tair_gaf.csv",
    index_col=0,
)
go_term_matrix

,GO:0000001,GO:0000009,GO:0000012,GO:0000014,GO:0000023,GO:0000024,GO:0000025,GO:0000026,GO:0000027,GO:0000028,...,GO:2001071,GO:2001080,GO:2001141,GO:2001142,GO:2001143,GO:2001147,GO:2001227,GO:2001280,GO:2001289,GO:2001294
GeneID,,,,,,,,,,,,,,,,,,,,,
AT1G01010,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT1G01020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT1G01030,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT1G01040,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT1G01046,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
AT5G67600,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT5G67610,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT5G67620,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
namespace_columns = [
    "biological_process",
    "molecular_function",
    "cellular_component",
]

graph = gobasic["graphs"][0]
records = []

for go_term in graph.get("nodes", []):
    meta = go_term.get("meta", {})
    basic_property_values = meta.get("basicPropertyValues", [])
    namespace_value = None

    for property_value in basic_property_values:
        if property_value.get("pred", "").endswith("#hasOBONamespace"):
            namespace_value = property_value.get("val")
            break

    go_identifier = go_term.get("id", "")
    go_number = go_identifier.rsplit("/", 1)[-1].replace("GO_", "")
    if not go_number.isdigit():
        continue

    formatted_go_id = f"GO:{int(go_number):07d}"
    record = {column: 0 for column in namespace_columns}
    if namespace_value in record:
        record[namespace_value] = 1
    records.append((formatted_go_id, record))

go_namespace_df = pd.DataFrame(
    [record for _, record in records],
    index=[go_id for go_id, _ in records],
)
go_namespace_df.index.name = "go_id"
go_namespace_df = go_namespace_df.sort_index()
go_namespace_df

,biological_process,molecular_function,cellular_component
go_id,,,
GO:0000001,1,0,0
GO:0000002,1,0,0
GO:0000003,1,0,0
GO:0000004,0,0,0
GO:0000005,0,1,0
...,...,...,...
GO:7770072,0,1,0
GO:7770073,0,1,0
GO:7770074,1,0,0


In [5]:
biological_process_go_terms = go_namespace_df[
    go_namespace_df["biological_process"] == 1
].index
molecular_function_go_terms = go_namespace_df[
    go_namespace_df["molecular_function"] == 1
].index

matrix_biological_process = go_term_matrix.loc[
    :, go_term_matrix.columns.isin(biological_process_go_terms)
]
matrix_molecular_function = go_term_matrix.loc[
    :, go_term_matrix.columns.isin(molecular_function_go_terms)
]
matrix_molecular_function
cellular_component_go_terms = go_namespace_df[
    go_namespace_df["cellular_component"] == 1
].index
matrix_cellular_component = go_term_matrix.loc[
    :, go_term_matrix.columns.isin(cellular_component_go_terms)
]

In [6]:
def read_gene_list(path):
    try:
        return pd.read_csv(path, header=None)[0].tolist()
    except pd.errors.EmptyDataError:
        return []


gene_list = read_gene_list(
    "/Users/michael/Data/Luke_terrace_experiment/General_data/Generated_data/deseq_dysbiosis_gene_list_recalled_genera.csv"
)
background_genes_list = read_gene_list(
    "/Users/michael/Data/Luke_terrace_experiment/General_data/Generated_data/deseq_background_genes_recalled_genera.csv"
)
print(f"Significant DE genes: {len(gene_list)}")
print(f"Background genes: {len(background_genes_list)}")

Significant DE genes: 0
Background genes: 18032


In [7]:
go_term_names = pd.read_csv(
    "/Users/michael/Data/Reference_data/goterm_names.csv", index_col=0
)
go_term_names

,x
GO:0000001,mitochondrion inheritance
GO:0000006,high-affinity zinc transmembrane transporter a...
GO:0000007,low-affinity zinc ion transmembrane transporte...
GO:0000009,"alpha-1,6-mannosyltransferase activity"
GO:0000010,heptaprenyl diphosphate synthase activity
...,...
GO:7770007,L-arginine conjugated cholate hydrolase activity
GO:7770008,L-histidine conjugated cholate hydrolase activity
GO:7770009,L-tryptophan conjugated cholate hydrolase acti...
GO:7770010,GTPBP3-MTO1 complex


In [8]:
def run_namespace_enrichment(go_annotations):
    """Fisher's exact GO enrichment of gene_list against background_genes_list."""
    if len(gene_list) == 0:
        print("No significant DE genes: skipping enrichment.")
        return pd.DataFrame(columns=["P_value", "x", "Neg Log P Val"])

    cont_tables = Go_tools.generate_contigency_tables(
        go_annotations=go_annotations,
        gene_list=gene_list,
        use_background_genes=True,
        background_genes=background_genes_list,
    )
    fishers_results = Go_tools.fishers_exact_on_contigency_tables(
        all_go_contigency_tables=cont_tables,
        original_GO_term_panda=go_annotations,
    )
    corrected_results = Go_tools.multi_hypothesis_correct_fishers_exact(fishers_results)
    named_results = corrected_results.merge(
        go_term_names, left_index=True, right_index=True, how="left"
    )
    named_results["x"] = named_results["x"].str.capitalize()
    named_results["Neg Log P Val"] = -1 * np.log10(named_results["P_value"])
    return named_results.sort_values(by="Neg Log P Val", ascending=False)


def plot_top_terms(named_results, n_terms, color):
    if named_results.empty:
        print("Nothing to plot.")
        return
    fig, ax = plt.subplots(figsize=(5, 10))
    fig.patch.set_facecolor("white")
    g = sns.barplot(
        data=named_results.head(n_terms),
        x="Neg Log P Val",
        y="x",
        color=color,
    )
    plt.xlabel("-log10(p value)", fontsize=14)
    g.set(ylabel=None)
    g.bar_label(g.containers[0], fmt="%.2f", label_type="center")
    g.grid(False)
    for side in ["bottom", "left", "top", "right"]:
        g.spines[side].set_color("black")
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.show()

## Biological process

In [9]:
named_biological_process_results = run_namespace_enrichment(matrix_biological_process)
named_biological_process_results.head(40)

No significant DE genes: skipping enrichment.


,P_value,x,Neg Log P Val


In [10]:
plot_top_terms(named_biological_process_results, 10, "tab:green")

Nothing to plot.


## Molecular function

In [11]:
named_molecular_function_results = run_namespace_enrichment(matrix_molecular_function)
named_molecular_function_results.head(40)

No significant DE genes: skipping enrichment.


,P_value,x,Neg Log P Val


In [12]:
plot_top_terms(named_molecular_function_results, 5, "tab:orange")

Nothing to plot.


## Cellular component

In [13]:
named_cellular_component_results = run_namespace_enrichment(matrix_cellular_component)
named_cellular_component_results.head(40)

No significant DE genes: skipping enrichment.


,P_value,x,Neg Log P Val


In [14]:
plot_top_terms(named_cellular_component_results, 10, "tab:purple")

Nothing to plot.
